In [2]:
import os
import sys
from pathlib import Path
import shutil
sys.path.insert(0, '/home/mwalker/git/neoexchange/neoexchange') #point to top of file path
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'neox.settings')
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')


import django
django.setup()
from django.conf import settings
from django.contrib.auth.models import User


from datetime import datetime, timedelta

os.chdir("/home/mwalker/git/neoexchange/neoexchange") #temp fix for json issue

from core.models import Frame, Block, StaticSource
from core.views import schedule_submit, record_block




In [6]:
try:
    ref_field_type = StaticSource.REFERENCE_FIELD
except AttributeError:
    ref_field_type = 16
coj_fields = StaticSource.objects.filter(source_type=ref_field_type, name__contains='COJ 2026 Field')   # change this to loop over bad fields: 15, 14, 13
site = 'coj'      # or missing fields: 0-5, 7, 10,
username = 'tlister@lcogt.net'
user = User.objects.get(username=username)
proposal = 'LCO2026A-003'
num_exps = 9

# Scheduling when all reference fields are defined (run once per night)

obs_date = datetime(2026, 7, 2)
try_num = 16 # Increment this each day  
for field in coj_fields.order_by('id'):
    print(f"Working on Field #{field.name} for {site.upper()}: ", end='')
    data = {'ra_deg' : field.ra, 'dec_deg' :  field.dec, 'source_id' : field.name,
            'slot_length' : 15,
            'exp_count' : num_exps, 'exp_length' : 60, 'gp_explength' : 60,  'rp_explength' : 60,  'ip_explength' : 60,  'zp_explength' : 60,
            'filter_pattern' : 'gp',
            'site' : site.lower(), 'site_code' : 'E10',
            'start_time' : obs_date, 'end_time' : obs_date+timedelta(seconds=86400-1),
            'user_id' : username, 'group_name' : field.name + f" try{try_num}",
            'proposal_code' : proposal,
            'max_airmass' : 1.5, 'min_lunar_dist' : 40,
            'bin_mode' : None, 'muscat_sync' : False, 'instrument_code' : '', 'period' : None, 'jitter' : None}
    observed_blocks = Block.objects.filter(calibsource=field, num_observed__gte=1)
    good_frames = Frame.objects.filter(block__in=observed_blocks, frametype=Frame.BANZAI_RED_FRAMETYPE).exclude(quality__contains=Frame.QUALITY_STREAKED)
    num_frames = good_frames.count()
    num_good_blocks = good_frames.values('block').distinct().count()
    expected_num_frames = max(num_good_blocks, 1) * num_exps * 4
    print(f"{num_frames}/{expected_num_frames} obtained. Done?:  {num_frames >= expected_num_frames}")
    

    
    if num_frames < expected_num_frames:
        print("Scheduling")
        tracking_num, sched_params = schedule_submit(data, field, data['user_id'])
#        tracking_num = None
    else:
        print("Already observed")
        tracking_num = None
   
    if tracking_num is not None:
        try:
            block_resp = record_block(tracking_num, sched_params, data, field, user)
        except TypeError:
            block_resp = record_block(tracking_num, sched_params, data, field)
        print(f"Created Block {tracking_num:} ? {block_resp:}")


Working on Field #Didymos COJ 2026 Field #01 for COJ: 0/36 obtained. Done?:  False
Scheduling


ERROR [02/Jul/2026 16:15:01] {'requests': [{'non_field_errors': ['According to the constraints of the request, the target will not be visible within the time window. Check that the target is in the nighttime sky. Consider modifying the time window or loosening the airmass or lunar separation constraints. If the target is non sidereal, double check that the provided elements are correct.']}]}
ERROR [02/Jul/2026 16:15:01] {'requests': [{'non_field_errors': ['According to the constraints of the request, the target will not be visible within the time window. Check that the target is in the nighttime sky. Consider modifying the time window or loosening the airmass or lunar separation constraints. If the target is non sidereal, double check that the provided elements are correct.']}]}


Created Block False ? False
Working on Field #Didymos COJ 2026 Field #02 for COJ: 0/36 obtained. Done?:  False
Scheduling


ERROR [02/Jul/2026 16:15:01] {'requests': [{'non_field_errors': ['According to the constraints of the request, the target will not be visible within the time window. Check that the target is in the nighttime sky. Consider modifying the time window or loosening the airmass or lunar separation constraints. If the target is non sidereal, double check that the provided elements are correct.']}]}
ERROR [02/Jul/2026 16:15:01] {'requests': [{'non_field_errors': ['According to the constraints of the request, the target will not be visible within the time window. Check that the target is in the nighttime sky. Consider modifying the time window or loosening the airmass or lunar separation constraints. If the target is non sidereal, double check that the provided elements are correct.']}]}


Created Block False ? False
Working on Field #Didymos COJ 2026 Field #03 for COJ: 0/36 obtained. Done?:  False
Scheduling


...

Created Block False ? False
Working on Field #Didymos COJ 2026 Field #08 for COJ: 72/72 obtained. Done?:  True
Already observed
Working on Field #Didymos COJ 2026 Field #09 for COJ: 36/36 obtained. Done?:  True
Already observed
Working on Field #Didymos COJ 2026 Field #10 for COJ: 0/36 obtained. Done?:  False
Scheduling


ERROR [02/Jul/2026 16:15:07] {'requests': [{'non_field_errors': ['According to the constraints of the request, the target will not be visible within the time window. Check that the target is in the nighttime sky. Consider modifying the time window or loosening the airmass or lunar separation constraints. If the target is non sidereal, double check that the provided elements are correct.']}]}
ERROR [02/Jul/2026 16:15:07] {'requests': [{'non_field_errors': ['According to the constraints of the request, the target will not be visible within the time window. Check that the target is in the nighttime sky. Consider modifying the time window or loosening the airmass or lunar separation constraints. If the target is non sidereal, double check that the provided elements are correct.']}]}


Created Block False ? False
Working on Field #Didymos COJ 2026 Field #11 for COJ: 45/108 obtained. Done?:  False
Scheduling


ERROR [02/Jul/2026 16:15:08] {'requests': [{'non_field_errors': ['According to the constraints of the request, the target will not be visible within the time window. Check that the target is in the nighttime sky. Consider modifying the time window or loosening the airmass or lunar separation constraints. If the target is non sidereal, double check that the provided elements are correct.']}]}
ERROR [02/Jul/2026 16:15:08] {'requests': [{'non_field_errors': ['According to the constraints of the request, the target will not be visible within the time window. Check that the target is in the nighttime sky. Consider modifying the time window or loosening the airmass or lunar separation constraints. If the target is non sidereal, double check that the provided elements are correct.']}]}


Created Block False ? False
Working on Field #Didymos COJ 2026 Field #12 for COJ: 36/36 obtained. Done?:  True
Already observed
Working on Field #Didymos COJ 2026 Field #13 for COJ: 36/36 obtained. Done?:  True
Already observed
Working on Field #Didymos COJ 2026 Field #14 for COJ: 40/36 obtained. Done?:  True
Already observed
Working on Field #Didymos COJ 2026 Field #15 for COJ: 72/72 obtained. Done?:  True
Already observed
Working on Field #Didymos COJ 2026 Field #16 for COJ: 36/36 obtained. Done?:  True
Already observed
